In [1]:
import AILibs
import numpy

import torch

dataset_root_path = "/users/michal/datasets/FordA/"

dataset_train   = AILibs.datasets.FordDataset(dataset_root_path, split='TRAIN')
dataset_test    = AILibs.datasets.FordDataset(dataset_root_path, split='TEST')

num_classes = dataset_train.num_classes
print(f"Number of classes: {num_classes}")  


def get_batch(dataset, batch_size=32):
    indices = numpy.random.choice(len(dataset), batch_size, replace=False)
    x_batch = torch.tensor(dataset.features[indices], dtype=torch.float32)
    y_batch = torch.tensor(dataset.labels[indices], dtype=torch.long)

    #x_batch = (x_batch - x_batch.mean(dim=1, keepdim=True))/(x_batch.std(dim=1, keepdim=True) + 1e-8)
    return x_batch, y_batch


Loading FordA TRAIN split...
Loaded 3601 samples.
Feature shape per sample: (500, 1) (seq_length, num_features)
Number of unique classes found: 2
Loading FordA TEST split...
Loaded 1320 samples.
Feature shape per sample: (500, 1) (seq_length, num_features)
Number of unique classes found: 2
Number of classes: 2


In [2]:
class RnnModel(torch.nn.Module):
  def __init__(self, num_inputs, num_outputs, num_hidden):
    super(RnnModel, self).__init__()

    # low level features, us 1D CNN, and stride to reduce sequence length and capture local patterns
    kernel_size = 7
    
    self.cnn_0  = torch.nn.Conv1d(in_channels=num_inputs, out_channels=num_hidden//2, kernel_size=kernel_size, padding=kernel_size//2, stride=4)
    self.act0   = torch.nn.SiLU()
    self.cnn_1  = torch.nn.Conv1d(in_channels=num_hidden//2, out_channels=num_hidden//2, kernel_size=kernel_size, padding=kernel_size//2, stride=4)
    self.act1   = torch.nn.SiLU()     

    self.rnn    = torch.nn.GRU(num_hidden//2, num_hidden, batch_first=True)
    self.fc     = torch.nn.Linear(num_hidden, num_outputs)

    torch.nn.init.orthogonal_(self.cnn_0.weight, gain=0.5)
    torch.nn.init.orthogonal_(self.cnn_1.weight, gain=0.5)
    torch.nn.init.orthogonal_(self.rnn.weight_ih_l0, gain=0.5)
    torch.nn.init.orthogonal_(self.rnn.weight_hh_l0, gain=0.5)

    torch.nn.init.zeros_(self.cnn_0.bias)
    torch.nn.init.zeros_(self.cnn_1.bias)
    torch.nn.init.zeros_(self.rnn.bias_ih_l0)
    torch.nn.init.zeros_(self.rnn.bias_hh_l0)
  
  def forward(self, x):

    # CNN for local features
    x = torch.transpose(x, 1, 2)  # (batch_size, num_inputs, seq_length)
    x = self.cnn_0(x)
    x = self.act0(x)
    x = self.cnn_1(x) 
    x = self.act1(x)
    x = torch.transpose(x, 1, 2)  # (batch_size, seq_length, num_hidden)

    rnn_output, hidden = self.rnn(x)  
    output = self.fc(rnn_output[:, -1, :])
    return output
    

In [3]:

num_inputs  = dataset_train.input_shape[1]
num_outputs = dataset_train.num_classes
num_hidden  = 128

model = RnnModel(num_inputs=num_inputs, num_outputs=num_outputs, num_hidden=num_hidden)

print(model)
    

optimizer = torch.optim.Adam(model.parameters(), lr=0.0025)

    
for n in range(2500):
    x_batch, y_batch = get_batch(dataset_train, batch_size=64)
    
    y_pred = model(x_batch) 
    loss = torch.nn.functional.cross_entropy(y_pred, y_batch)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f"Step {n}, Loss: {loss.item()}")



RnnModel(
  (cnn_0): Conv1d(1, 64, kernel_size=(7,), stride=(4,), padding=(3,))
  (act0): SiLU()
  (cnn_1): Conv1d(64, 64, kernel_size=(7,), stride=(4,), padding=(3,))
  (act1): SiLU()
  (rnn): GRU(64, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)
Step 0, Loss: 0.6929799914360046
Step 1, Loss: 0.6945866942405701
Step 2, Loss: 0.6948250532150269
Step 3, Loss: 0.6928185224533081
Step 4, Loss: 0.6931174397468567
Step 5, Loss: 0.6925441026687622
Step 6, Loss: 0.690601646900177
Step 7, Loss: 0.6927450895309448
Step 8, Loss: 0.6879749298095703
Step 9, Loss: 0.6894404888153076
Step 10, Loss: 0.6967401504516602
Step 11, Loss: 0.6979329586029053
Step 12, Loss: 0.6847710013389587
Step 13, Loss: 0.6971113085746765
Step 14, Loss: 0.6984586715698242
Step 15, Loss: 0.6883851885795593
Step 16, Loss: 0.6892675161361694
Step 17, Loss: 0.6960020065307617
Step 18, Loss: 0.6921202540397644
Step 19, Loss: 0.6924389600753784
Step 20, Loss: 0.6808878183364868
Step 21, Lo

In [4]:

print("Predicting with RNN...")

y_test    = []
y_pred    = []
  
for n in range(len(dataset_test)):
    y_gt = dataset_test.labels[n]
    
    x_batch = torch.tensor(dataset_test.features[n], dtype=torch.float32)
    x_batch = x_batch.unsqueeze(0)  # Add batch dimension
    with torch.no_grad():
        y = model(x_batch)
        y = y.squeeze(0).detach().numpy()
    
    y_test.append(y_gt)
    y_pred.append(y)
    
y_test = numpy.array(y_test)
y_pred = numpy.array(y_pred)


metrics = AILibs.metrics.classification_evaluation(y_test, y_pred, num_classes)


for key, value in metrics.items():
    print(f"{key}: {value}")

    

Predicting with RNN...
n_samples: 1320
num_classes: 2
accuracy: 0.93258
macro_precision: 0.93245
macro_recall: 0.93282
macro_f1_score: 0.93254
macro_mcc: 0.86527
macro_specificity: 0.93282
macro_balanced_accuracy: 0.93282
macro_iou: 0.87362
macro_dice: 0.93254
tp_per_class: [630, 601]
tn_per_class: [601, 630]
fp_per_class: [38, 51]
fn_per_class: [51, 38]
precision_per_class: [0.94311, 0.92178]
recall_per_class: [0.92511, 0.94053]
f1_score_per_class: [0.93403, 0.93106]
